# Multimodal Histopathology XAI Framework — BreaKHis
**Vision-XAI-Report Pipeline**: ConvNeXt-Base + Swin-Base + 6 XAI methods

### Steps
1. Mount Google Drive & set up project
2. Install dependencies
3. Download BreaKHis dataset from Kaggle
4. Run data pipeline (split)
5. Train model
6. Run XAI inference
7. Evaluate

In [ ]:
# ── Cell 1: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this to where your project is in Drive, or leave as-is to use Colab local storage
PROJECT_ROOT = '/content/breakhis_xai'
os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Cell 2: Upload project files from Drive OR zip ──
# Option A: If you uploaded a zip to Drive
# DRIVE_ZIP = '/content/drive/MyDrive/NewImplementation.zip'
# !unzip -q "$DRIVE_ZIP" -d "$PROJECT_ROOT"

# Option B: Upload kaggle.json for dataset download
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # upload kaggle.json

import os, json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(uploaded_path := list(uploaded.keys())[0], 'rb') as f:
    creds = json.loads(f.read())
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json configured.')

In [ ]:
# ── Cell 3: Clone / copy project files ──
# If your project is in Google Drive:
DRIVE_PROJECT = '/content/drive/MyDrive/Thesis/New Implementation'

import shutil
if os.path.exists(DRIVE_PROJECT):
    if os.path.exists(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)
    shutil.copytree(DRIVE_PROJECT, PROJECT_ROOT)
    print(f'Copied project from Drive to {PROJECT_ROOT}')
else:
    print(f'Drive path not found: {DRIVE_PROJECT}')
    print('Please upload your project zip or adjust DRIVE_PROJECT path.')

os.listdir(PROJECT_ROOT)

In [ ]:
# ── Cell 4: Install dependencies ──
!pip install -q \
    torch torchvision timm \
    transformers accelerate bitsandbytes \
    captum shap lime \
    scikit-learn rouge-score \
    matplotlib seaborn wandb \
    Pillow numpy tqdm opencv-python \
    kagglehub

print('All dependencies installed.')

In [ ]:
# ── Cell 5: Verify GPU ──
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 6: Add project to Python path ──
import sys
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 7: Download BreaKHis dataset from Kaggle ──
import kagglehub
path = kagglehub.dataset_download('ambarish/breakhis')
print(f'Dataset downloaded to: {path}')

# Set dataset path for the pipeline
DATASET_RAW = path
print('Files:', os.listdir(path)[:5])

In [ ]:
# ── Cell 8: Patch config to use Colab paths ──
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'config', 'settings.py')

with open(CONFIG_PATH, 'r') as f:
    config = f.read()

# Override DATA_DIR to point to Colab dataset location
DATASETS_DIR = '/content/datasets'
os.makedirs(DATASETS_DIR, exist_ok=True)

config = config.replace(
    "DATA_DIR = BASE_DIR.parent / \"datasets\"",
    f'DATA_DIR = Path("{DATASETS_DIR}")'
)

with open(CONFIG_PATH, 'w') as f:
    f.write(config)

print('Config patched for Colab paths.')

In [ ]:
# ── Cell 9: Run data pipeline (split only, skip download) ──
!python run_data_pipeline.py --skip-download --skip-eda 2>&1 | tail -40

In [ ]:
# ── Cell 10: Train model (GPU, 50 epochs, full dataset) ──
# For a quick test run use: --subset 800 --epochs 2
# For full training use: --epochs 50 (no --subset)

!python run_train.py \
    --task multiclass \
    --magnification 400X \
    --epochs 50 \
    --batch-size 16 \
    --no-wandb \
    2>&1 | tee /content/drive/MyDrive/Thesis/training_log.txt

In [ ]:
# ── Cell 11: Copy checkpoint to Drive ──
CKPT_SRC = os.path.join(PROJECT_ROOT, 'outputs', 'checkpoints', 'best_model.pth')
CKPT_DST = '/content/drive/MyDrive/Thesis/best_model.pth'

if os.path.exists(CKPT_SRC):
    shutil.copy2(CKPT_SRC, CKPT_DST)
    print(f'Checkpoint saved to Drive: {CKPT_DST}')
else:
    print('Checkpoint not found — check training output above.')

In [ ]:
# ── Cell 12: Run XAI inference on a sample image ──
# Find a sample image from the test set
import glob
test_images = glob.glob(f'{DATASETS_DIR}/BreaKHis 400X/test/**/*.png', recursive=True)
print(f'Found {len(test_images)} test images')

SAMPLE_IMAGE = test_images[0] if test_images else None
CHECKPOINT   = os.path.join(PROJECT_ROOT, 'outputs', 'checkpoints', 'best_model.pth')

if SAMPLE_IMAGE and os.path.exists(CHECKPOINT):
    print(f'Running inference on: {SAMPLE_IMAGE}')
    !python run_inference.py \
        --checkpoint "$CHECKPOINT" \
        --image "$SAMPLE_IMAGE" \
        --task multiclass \
        2>&1
else:
    print('Missing checkpoint or test images.')

In [ ]:
# ── Cell 13: Display XAI panel ──
from IPython.display import Image as IPImage, display
import glob

xai_panels = glob.glob(os.path.join(PROJECT_ROOT, 'outputs', 'xai_visualizations', '*_xai_panel.png'))
for panel in xai_panels[:3]:
    print(panel)
    display(IPImage(panel))

In [ ]:
# ── Cell 14: Run evaluation ──
CHECKPOINT = os.path.join(PROJECT_ROOT, 'outputs', 'checkpoints', 'best_model.pth')

!python run_evaluate.py \
    --checkpoint "$CHECKPOINT" \
    --n-samples 50 \
    2>&1

In [ ]:
# ── Cell 15: Copy all outputs to Drive ──
OUTPUTS_SRC = os.path.join(PROJECT_ROOT, 'outputs')
OUTPUTS_DST = '/content/drive/MyDrive/Thesis/outputs'

if os.path.exists(OUTPUTS_DST):
    shutil.rmtree(OUTPUTS_DST)
shutil.copytree(OUTPUTS_SRC, OUTPUTS_DST)
print(f'All outputs saved to Drive: {OUTPUTS_DST}')